In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from category_encoders import TargetEncoder
import optuna

## Manual Parameter Tuning with PCA

In [2]:
# Load the training and test data
train2 = pd.read_csv('train_feature_v1.csv')
train2 = train2.rename(columns=lambda x: x.replace('[^A-Za-z0-9_]+', ''))
test2 = pd.read_csv('test_feature_v1.csv')
test2 = test2.rename(columns=lambda x: x.replace('[^A-Za-z0-9_]+', ''))

In [3]:
# Perform Principal Component Analysis (PCA) on selected features
pca_cols_train = train2.iloc[:, 5:19]
pca_cols_train = pca_cols_train.apply(LabelEncoder().fit_transform)
pca = PCA(n_components=5)
pca_result = pca.fit_transform(pca_cols_train)
pca_result = pd.DataFrame(pca_result, columns=[f'pca_{i}' for i in range(5)])
train2 = train2.drop(columns=pca_cols_train.columns)
train2 = pd.concat([train2, pca_result], axis=1)

pca_cols_test = test2.iloc[:, 5:19]
pca_cols_test = pca_cols_test.apply(LabelEncoder().fit_transform)
pca_result = pca.fit_transform(pca_cols_test)
pca_result = pd.DataFrame(pca_result, columns=[f'pca_{i}' for i in range(5)])
test2 = test2.drop(columns=pca_cols_train.columns)
test2 = pd.concat([test2, pca_result], axis=1)

In [4]:
# Separate features and target variables
X_train = train2.drop(columns=['price'])
y_train = train2[['price']]

In [5]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [6]:
# Set the LightGBM hyperparameters
hyper_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.2,
    'verbose': -1,
    'max_depth': 5,
    'num_leaves': 31,
    'subsample': 0.8,
    'n_estimators': 1000,
    'colsample_bytree': 0.9,
    'reg_lambda': 0.2,
    'early_stopping_rounds': 50
}


In [7]:
# Train the LightGBM model
gbm = lgb.LGBMRegressor(**hyper_params)
gbm.fit(X_train, y_train, eval_set=[(X_test, y_test)])


[LightGBM] [Warning] early_stopping_round is set=50, early_stopping_rounds=50 will be ignored. Current value: early_stopping_round=50
[1]	valid_0's rmse: 117672
[2]	valid_0's rmse: 97692.9
[3]	valid_0's rmse: 82036.1
[4]	valid_0's rmse: 69370.5
[5]	valid_0's rmse: 59281.2
[6]	valid_0's rmse: 51758.9
[7]	valid_0's rmse: 45650.2
[8]	valid_0's rmse: 41374.8
[9]	valid_0's rmse: 37394
[10]	valid_0's rmse: 34548.8
[11]	valid_0's rmse: 32676.2
[12]	valid_0's rmse: 31043.1
[13]	valid_0's rmse: 30010.3
[14]	valid_0's rmse: 29140.6
[15]	valid_0's rmse: 28679.6
[16]	valid_0's rmse: 28344.6
[17]	valid_0's rmse: 27994.9
[18]	valid_0's rmse: 27907.9
[19]	valid_0's rmse: 27563
[20]	valid_0's rmse: 27388.2
[21]	valid_0's rmse: 27297.1
[22]	valid_0's rmse: 27141.8
[23]	valid_0's rmse: 26922
[24]	valid_0's rmse: 26689.1
[25]	valid_0's rmse: 26509
[26]	valid_0's rmse: 26218.4
[27]	valid_0's rmse: 26270.2
[28]	valid_0's rmse: 26254.6
[29]	valid_0's rmse: 26166.3
[30]	valid_0's rmse: 26132.3
[31]	valid_0's

LGBMRegressor(colsample_bytree=0.9, early_stopping_rounds=50, learning_rate=0.2,
              max_depth=5, metric='rmse', n_estimators=1000,
              objective='regression', reg_lambda=0.2, subsample=0.8,
              verbose=-1)

In [8]:
# Perform label encoding and target encoding
train_label_encoded = pd.read_csv('train_feature_label_encoded.csv')
test_label_encoded = pd.read_csv('test_feature_label_encoded.csv')

X_train_label_encoded = train_label_encoded.drop(columns=['price'])
y_train_label_encoded = train_label_encoded[['price']]
X_train_label_encoded, X_test_label_encoded, y_train_label_encoded, y_test_label_encoded = train_test_split(X_train_label_encoded, y_train_label_encoded, test_size=0.2, random_state=42)


## Experiment with simple encoding of categorical features

In [9]:
train_label_encoded = pd.read_csv('train_feature_label_encoded.csv')
test_label_encoded = pd.read_csv('test_feature_label_encoded.csv')

# Separate features and target
X = train_label_encoded[['feature_make_imputed']]
y = train_label_encoded['price']

# Initialize and fit TargetEncoder
encoder = TargetEncoder()
X_encoded = encoder.fit_transform(X, y)

# Assign the encoded values back to the DataFrame
train_label_encoded['feature_make_imputed'] = X_encoded

# Assign the encoded values back to the DataFrame
test_label_encoded['feature_make_imputed'] = encoder.transform(test_label_encoded['feature_make_imputed'])


X_train_label_encoded = train_label_encoded.drop(columns=['price'])
Y_train_label_encoded = train_label_encoded[['price']]

X_train_label_encoded, X_test_label_encoded, y_train_label_encoded, y_test_label_encoded = train_test_split(X_train_label_encoded, Y_train_label_encoded, test_size=0.2, random_state=42)

In [10]:
# Define the categorical features for LightGBM
categorical_features = ["feature_is_electric", "feature_make_imputed", "feature_make_model_target_encoded", "feature_fuel_label", "feature_low_mileage", "feature_type", "feature_ownership", "feature_condition", "feature_special_features", "feature_sales", "feature_price_category", "feature_type_of_vehicle"]

In [11]:
# Train the LightGBM model with label encoded data
hyper_params_label_encoded = hyper_params.copy()
gbm_label_encoded = lgb.LGBMRegressor(**hyper_params_label_encoded)
gbm_label_encoded.fit(X_train_label_encoded, y_train_label_encoded, eval_set=[(X_test_label_encoded, y_test_label_encoded)], categorical_feature=categorical_features)

C:\Users\tan_l\anaconda3\envs\PyTorch\Lib\site-packages\lightgbm\basic.py:2065: UserWarning: Using categorical_feature in Dataset.
  _log_warning('Using categorical_feature in Dataset.')
C:\Users\tan_l\anaconda3\envs\PyTorch\Lib\site-packages\lightgbm\basic.py:2068: UserWarning: categorical_feature in Dataset is overridden.
New categorical_feature is ['feature_condition', 'feature_fuel_label', 'feature_is_electric', 'feature_low_mileage', 'feature_make_imputed', 'feature_make_model_target_encoded', 'feature_ownership', 'feature_price_category', 'feature_sales', 'feature_special_features', 'feature_type', 'feature_type_of_vehicle']
  _log_warning('categorical_feature in Dataset is overridden.\n'
C:\Users\tan_l\anaconda3\envs\PyTorch\Lib\site-packages\lightgbm\basic.py:1780: UserWarning: Overriding the parameters from Reference Dataset.
  _log_warning('Overriding the parameters from Reference Dataset.')
C:\Users\tan_l\anaconda3\envs\PyTorch\Lib\site-packages\lightgbm\basic.py:1513: UserW

[1]	valid_0's rmse: 117465
[2]	valid_0's rmse: 96880.5
[3]	valid_0's rmse: 80875
[4]	valid_0's rmse: 68368.1
[5]	valid_0's rmse: 58966.2
[6]	valid_0's rmse: 51768.6
[7]	valid_0's rmse: 45704.3
[8]	valid_0's rmse: 41656.3
[9]	valid_0's rmse: 37477.3
[10]	valid_0's rmse: 34471.1
[11]	valid_0's rmse: 32396.3
[12]	valid_0's rmse: 31101.5
[13]	valid_0's rmse: 29984
[14]	valid_0's rmse: 29358.2
[15]	valid_0's rmse: 28918.8
[16]	valid_0's rmse: 28561.5
[17]	valid_0's rmse: 28229.2
[18]	valid_0's rmse: 28055.6
[19]	valid_0's rmse: 27928.6
[20]	valid_0's rmse: 27852.1
[21]	valid_0's rmse: 27758.2
[22]	valid_0's rmse: 27642.4
[23]	valid_0's rmse: 27712.2
[24]	valid_0's rmse: 27520.2
[25]	valid_0's rmse: 27441.6
[26]	valid_0's rmse: 27312.7
[27]	valid_0's rmse: 27137.8
[28]	valid_0's rmse: 27126.3
[29]	valid_0's rmse: 27051.2
[30]	valid_0's rmse: 27090.1
[31]	valid_0's rmse: 27058.3
[32]	valid_0's rmse: 27040.2
[33]	valid_0's rmse: 27030
[34]	valid_0's rmse: 27008.3
[35]	valid_0's rmse: 26794.4
[

LGBMRegressor(colsample_bytree=0.9, early_stopping_rounds=50, learning_rate=0.2,
              max_depth=5, metric='rmse', n_estimators=1000,
              objective='regression', reg_lambda=0.2, subsample=0.8,
              verbose=-1)

## Parameter tuning with Grid Search

In [12]:
# Perform grid search to find the best hyperparameters
param_grid = {
    'num_leaves': [31, 63, 127],
    'max_depth': [3, 5, 7],
    'min_data_in_leaf': [20, 50, 100],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300]
}

grid_search = GridSearchCV(estimator=lgb.LGBMRegressor(random_state=42), param_grid=param_grid, cv=5, scoring='neg_root_mean_squared_error', verbose=1)
grid_search.fit(X_train_label_encoded, y_train_label_encoded)
best_model = grid_search.best_estimator_
grid_best_model = best_model


Fitting 5 folds for each of 243 candidates, totalling 1215 fits
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be igno

## Parameter tuning with Grid Search

In [13]:
# Optimize the model using Optuna
def objective(trial):
    # Hyperparameter optimization
    num_leaves = trial.suggest_int('num_leaves', 31, 127)
    max_depth = trial.suggest_int('max_depth', 3, 7)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 20, 100)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.1)
    n_estimators = trial.suggest_int('n_estimators', 100, 300)
    model = lgb.LGBMRegressor(num_leaves=num_leaves, max_depth=max_depth, min_data_in_leaf=min_data_in_leaf, learning_rate=learning_rate, n_estimators=n_estimators, random_state=42)
    model.fit(X_train_label_encoded, y_train_label_encoded)
    rmse = np.sqrt(mean_squared_error(y_test_label_encoded, model.predict(X_test_label_encoded)))
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)
optuna_best_model = lgb.LGBMRegressor(**study.best_params)
optuna_best_model.fit(X_train_label_encoded, y_train_label_encoded)

[I 2024-11-14 00:40:39,783] A new study created in memory with name: no-name-cb00a5e6-50bb-4f2f-a186-9301ff115e3e
[I 2024-11-14 00:40:39,952] Trial 0 finished with value: 30001.94228280882 and parameters: {'num_leaves': 97, 'max_depth': 6, 'min_data_in_leaf': 80, 'learning_rate': 0.0528731895285836, 'n_estimators': 154}. Best is trial 0 with value: 30001.94228280882.
[I 2024-11-14 00:40:40,140] Trial 1 finished with value: 33629.08498889498 and parameters: {'num_leaves': 103, 'max_depth': 6, 'min_data_in_leaf': 72, 'learning_rate': 0.02690147091804672, 'n_estimators': 108}. Best is trial 0 with value: 30001.94228280882.


[LightGBM] [Warning] min_data_in_leaf is set=72, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=72


[I 2024-11-14 00:40:40,268] Trial 2 finished with value: 30588.6223350236 and parameters: {'num_leaves': 63, 'max_depth': 3, 'min_data_in_leaf': 56, 'learning_rate': 0.042334396947793146, 'n_estimators': 186}. Best is trial 0 with value: 30001.94228280882.


[LightGBM] [Warning] min_data_in_leaf is set=56, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=56
[LightGBM] [Warning] min_data_in_leaf is set=32, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=32


[I 2024-11-14 00:40:40,407] Trial 3 finished with value: 25996.17581164834 and parameters: {'num_leaves': 33, 'max_depth': 5, 'min_data_in_leaf': 32, 'learning_rate': 0.06225273987115385, 'n_estimators': 114}. Best is trial 3 with value: 25996.17581164834.


[LightGBM] [Warning] min_data_in_leaf is set=68, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=68


... (59 outputs left out)

[LightGBM] [Warning] min_data_in_leaf is set=74, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=74
[LightGBM] [Warning] min_data_in_leaf is set=23, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=23


[I 2024-11-14 00:40:50,908] Trial 48 finished with value: 46233.578476405 and parameters: {'num_leaves': 67, 'max_depth': 7, 'min_data_in_leaf': 23, 'learning_rate': 0.010401145367816306, 'n_estimators': 134}. Best is trial 13 with value: 22939.159388978456.
[I 2024-11-14 00:40:51,097] Trial 49 finished with value: 26629.020821125112 and parameters: {'num_leaves': 51, 'max_depth': 4, 'min_data_in_leaf': 49, 'learning_rate': 0.07830351514032634, 'n_estimators': 259}. Best is trial 13 with value: 22939.159388978456.


[LightGBM] [Warning] min_data_in_leaf is set=49, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=49
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20


LGBMRegressor(learning_rate=0.09942314328841917, max_depth=4,
              min_data_in_leaf=20, n_estimators=239, num_leaves=66)

## Model Evaluation

In [14]:
# Evaluate the GridSearch Best Model
print("GridSearch Best Model:")
grid_train_rmse = mean_squared_error(y_train_label_encoded, grid_best_model.predict(X_train_label_encoded), squared=False)
grid_test_rmse = mean_squared_error(y_test_label_encoded, grid_best_model.predict(X_test_label_encoded), squared=False)
grid_train_mae = mean_absolute_error(y_train_label_encoded, grid_best_model.predict(X_train_label_encoded))
grid_test_mae = mean_absolute_error(y_test_label_encoded, grid_best_model.predict(X_test_label_encoded))
grid_train_r2 = r2_score(y_train_label_encoded, grid_best_model.predict(X_train_label_encoded))
grid_test_r2 = r2_score(y_test_label_encoded, grid_best_model.predict(X_test_label_encoded))
print(f"Train RMSE: {grid_train_rmse:.2f}, Test RMSE: {grid_test_rmse:.2f}")
print(f"Train MAE: {grid_train_mae:.2f}, Test MAE: {grid_test_mae:.2f}")
print(f"Train R-squared: {grid_train_r2:.2f}, Test R-squared: {grid_test_r2:.2f}")

# Evaluate the Optuna Best Model
print("\nOptuna Best Model:")
optuna_train_rmse = mean_squared_error(y_train_label_encoded, optuna_best_model.predict(X_train_label_encoded), squared=False)
optuna_test_rmse = mean_squared_error(y_test_label_encoded, optuna_best_model.predict(X_test_label_encoded), squared=False)
optuna_train_mae = mean_absolute_error(y_train_label_encoded, optuna_best_model.predict(X_train_label_encoded))
optuna_test_mae = mean_absolute_error(y_test_label_encoded, optuna_best_model.predict(X_test_label_encoded))
optuna_train_r2 = r2_score(y_train_label_encoded, optuna_best_model.predict(X_train_label_encoded))
optuna_test_r2 = r2_score(y_test_label_encoded, optuna_best_model.predict(X_test_label_encoded))
print(f"Train RMSE: {optuna_train_rmse:.2f}, Test RMSE: {optuna_test_rmse:.2f}")
print(f"Train MAE: {optuna_train_mae:.2f}, Test MAE: {optuna_test_mae:.2f}")
print(f"Train R-squared: {optuna_train_r2:.2f}, Test R-squared: {optuna_test_r2:.2f}")

# Evaluate the GBM Model
print("\nGBM Model:")
gbm_train_rmse = mean_squared_error(y_train, gbm.predict(X_train), squared=False)
gbm_test_rmse = mean_squared_error(y_test, gbm.predict(X_test), squared=False)
gbm_train_mae = mean_absolute_error(y_train, gbm.predict(X_train))
gbm_test_mae = mean_absolute_error(y_test, gbm.predict(X_test))
gbm_train_r2 = r2_score(y_train, gbm.predict(X_train))
gbm_test_r2 = r2_score(y_test, gbm.predict(X_test))
print(f"Train RMSE: {gbm_train_rmse:.2f}, Test RMSE: {gbm_test_rmse:.2f}")
print(f"Train MAE: {gbm_train_mae:.2f}, Test MAE: {gbm_test_mae:.2f}")
print(f"Train R-squared: {gbm_train_r2:.2f}, Test R-squared: {gbm_test_r2:.2f}")

# Evaluate the GBM Label Encoded Model
print("\nGBM Label Encoded Model:")
gbm_label_train_rmse = mean_squared_error(y_train_label_encoded, gbm_label_encoded.predict(X_train_label_encoded), squared=False)
gbm_label_test_rmse = mean_squared_error(y_test_label_encoded, gbm_label_encoded.predict(X_test_label_encoded), squared=False)
gbm_label_train_mae = mean_absolute_error(y_train_label_encoded, gbm_label_encoded.predict(X_train_label_encoded))
gbm_label_test_mae = mean_absolute_error(y_test_label_encoded, gbm_label_encoded.predict(X_test_label_encoded))
gbm_label_train_r2 = r2_score(y_train_label_encoded, gbm_label_encoded.predict(X_train_label_encoded))
gbm_label_test_r2 = r2_score(y_test_label_encoded, gbm_label_encoded.predict(X_test_label_encoded))
print(f"Train RMSE: {gbm_label_train_rmse:.2f}, Test RMSE: {gbm_label_test_rmse:.2f}")
print(f"Train MAE: {gbm_label_train_mae:.2f}, Test MAE: {gbm_label_test_mae:.2f}")
print(f"Train R-squared: {gbm_label_train_r2:.2f}, Test R-squared: {gbm_label_test_r2:.2f}")

GridSearch Best Model:
Train RMSE: 9703.46, Test RMSE: 22482.15
Train MAE: 4352.58, Test MAE: 5804.75
Train R-squared: 1.00, Test R-squared: 0.98

Optuna Best Model:
Train RMSE: 13018.86, Test RMSE: 22939.16
Train MAE: 5480.82, Test MAE: 6522.19
Train R-squared: 0.99, Test R-squared: 0.97

GBM Model:
Train RMSE: 5283.46, Test RMSE: 22474.26
Train MAE: 3000.22, Test MAE: 5386.74
Train R-squared: 1.00, Test R-squared: 0.98

GBM Label Encoded Model:
Train RMSE: 4895.03, Test RMSE: 24244.49
Train MAE: 2926.40, Test MAE: 5954.12
Train R-squared: 1.00, Test R-squared: 0.97


In [15]:
# Make predictions and export to CSV files
grid_preds = grid_best_model.predict(test_label_encoded)
optuna_preds = optuna_best_model.predict(test_label_encoded)
gbm_preds = gbm.predict(test2)
gbm_label_preds = gbm_label_encoded.predict(test_label_encoded)

pd.DataFrame({'Id': range(len(test_label_encoded)), 'Predicted': grid_preds}).to_csv('grid_search_predictions.csv', index=False)
pd.DataFrame({'Id': range(len(test_label_encoded)), 'Predicted': optuna_preds}).to_csv('optuna_predictions.csv', index=False)
pd.DataFrame({'Id': range(len(test2)), 'Predicted': gbm_preds}).to_csv('gbm_predictions.csv', index=False)
pd.DataFrame({'Id': range(len(test_label_encoded)), 'Predicted': gbm_label_preds}).to_csv('gbm_label_predictions.csv', index=False)

print("Predictions exported to separate CSV files.")

Predictions exported to separate CSV files.


In [17]:
# Save the LightGBM models
grid_best_model.booster_.save_model('grid_best_model.txt')
optuna_best_model.booster_.save_model('optuna_best_model.txt')
gbm.booster_.save_model('gbm_model.txt')
gbm_label_encoded.booster_.save_model('gbm_label_encoded_model.txt')

print("Models saved successfully.")

Models saved successfully.
